In [1]:
from google.colab import files
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.layout import Layout, ManualLayout
from statistics import median
from copy import copy

wgrane = files.upload()
nazwa_pliku = list(wgrane.keys())[0]

wb = load_workbook(nazwa_pliku)
ws = wb["Arkusz1"]

# DANE
WIERSZ_NAGLOWKA = 2     # nazwy kryteriów
WIERSZ_SD = 3           # litery S / D
WIERSZ_WAG = 4          # wagi
PIERWSZY_MENEDZER = 5   # pierwszy menedżer
OSTATNI_MENEDZER = 14   # ostatni menedżer
PIERWSZA_KOLUMNA = 2    # kolumna B
OSTATNIA_KOLUMNA = 16   # kolumna P
PRZESUNIECIE = 16       # o ile wierszy niżej wstawimy nową tabelę

menedzerowie = range(PIERWSZY_MENEDZER, OSTATNI_MENEDZER + 1)


# waluta na float
def na_liczbe(wartosc):
    if isinstance(wartosc, (int, float)):
        return wartosc
    tekst = str(wartosc).replace("USD", "").replace(",", ".").strip()
    return float(tekst)


# 0/1
wyniki = {}

for kol in range(PIERWSZA_KOLUMNA, OSTATNIA_KOLUMNA + 1):
    nazwa = ws.cell(WIERSZ_NAGLOWKA, kol).value
    typ = ws.cell(WIERSZ_SD, kol).value          # "S" albo "D"
    wartosci = [ws.cell(w, kol).value for w in menedzerowie]

    # puste
    if None in wartosci:
        raise ValueError("Pusta komórka w kryterium: " + str(nazwa) + ". Uzupełnij ją w Excelu.")

    # TAK/NIE
    if all(w in ("TAK", "NIE") for w in wartosci):
        for w in menedzerowie:
            if ws.cell(w, kol).value == "TAK":
                wynik = 1
            else:
                wynik = 0
            if typ == "D":
                wynik = 1 - wynik
            wyniki[(w, kol)] = wynik

    # skala 1-5 i ilościowa: wzór na przedział [0,1]
    else:
        # do min i max bierzemy tylko liczby (bez "Brak planu płatnego")
        liczby = []
        for v in wartosci:
            if not str(v).startswith("Brak"):
                liczby.append(na_liczbe(v))
        minimum = min(liczby)
        maksimum = max(liczby)
        print(nazwa, "-> min =", minimum, ", max =", maksimum)

        if maksimum == minimum:
            raise ValueError("W kryterium " + str(nazwa) + " wszystkie liczby są takie same, nie da się policzyć wzoru.")

        for w in menedzerowie:
            v = ws.cell(w, kol).value
            if str(v).startswith("Brak"):      # "Brak planu płatnego" = 1
                wynik = 1
            else:
                x = na_liczbe(v)
                if typ == "S":                 # stymulanta
                    wynik = (x - minimum) / (maksimum - minimum)
                else:                          # destymulanta
                    wynik = (maksimum - x) / (maksimum - minimum)
            wyniki[(w, kol)] = wynik


# tabelka
ws.cell(WIERSZ_NAGLOWKA - 1 + PRZESUNIECIE, 1).value = "DANE PO NORMALIZACJI ([0,1])"

for wiersz in (WIERSZ_NAGLOWKA, WIERSZ_SD, WIERSZ_WAG):
    for kol in range(1, OSTATNIA_KOLUMNA + 1):
        stara = ws.cell(wiersz, kol)
        nowa = ws.cell(wiersz + PRZESUNIECIE, kol)
        nowa.value = stara.value
        nowa._style = copy(stara._style)

for w in menedzerowie:
    stara = ws.cell(w, 1)
    nowa = ws.cell(w + PRZESUNIECIE, 1)
    nowa.value = stara.value
    nowa._style = copy(stara._style)
    for kol in range(PIERWSZA_KOLUMNA, OSTATNIA_KOLUMNA + 1):
        stara = ws.cell(w, kol)
        nowa = ws.cell(w + PRZESUNIECIE, kol)
        nowa.value = wyniki[(w, kol)]
        nowa._style = copy(stara._style)
        nowa.number_format = "0.00"


# kontrola: wagi powinny sumować się do 1
print("Suma wag:", round(sum(ws.cell(WIERSZ_WAG, k).value for k in range(PIERWSZA_KOLUMNA, OSTATNIA_KOLUMNA + 1)), 3))


# SUMA WAŻONA (każdy wynik razy waga kryterium, razy 100)
KOLUMNA_SUMY = OSTATNIA_KOLUMNA + 1    # kolumna Q

# nagłówek kolumny z sumą (taki sam wygląd jak pozostałe nagłówki)
naglowek = ws.cell(WIERSZ_NAGLOWKA + PRZESUNIECIE, KOLUMNA_SUMY)
naglowek.value = "Suma ważona (0-100)"
naglowek._style = copy(ws.cell(WIERSZ_NAGLOWKA + PRZESUNIECIE, OSTATNIA_KOLUMNA)._style)

sumy = {}   # sumy[nazwa menedżera] = suma jego jedynek
for w in menedzerowie:
    suma = 0
    for kol in range(PIERWSZA_KOLUMNA, OSTATNIA_KOLUMNA + 1):
        waga = ws.cell(WIERSZ_WAG, kol).value
        suma = suma + wyniki[(w, kol)] * waga
    suma = round(suma * 100, 1)     # skala 0-100, jak w prezentacji
    komorka = ws.cell(w + PRZESUNIECIE, KOLUMNA_SUMY)
    komorka.value = suma
    komorka._style = copy(ws.cell(w + PRZESUNIECIE, OSTATNIA_KOLUMNA)._style)
    # krótka nazwa do rankingu i wykresu (bez nawiasów i dopisków po ukośniku)
    krotka = str(ws.cell(w, 1).value).split(" (")[0].split(" /")[0].strip()
    sumy[krotka] = suma


# szerokość kolumny Q dopasowana do nagłówka
ws.column_dimensions["Q"].width = len("Suma ważona (0-100)") + 2


# RANKING (od największej sumy do najmniejszej)
ranking = sorted(sumy.items(), key=lambda para: para[1], reverse=True)

START = OSTATNI_MENEDZER + PRZESUNIECIE + 3     # wiersz, od którego zaczyna się ranking
ws.cell(START, 1).value = "RANKING MENEDŻERÓW"
ws.cell(START + 1, 1).value = "Menedżer"
ws.cell(START + 1, 2).value = "Suma ważona (0-100)"

wiersz = START + 2
for nazwa, suma in ranking:
    ws.cell(wiersz, 1).value = nazwa
    ws.cell(wiersz, 2).value = suma
    wiersz = wiersz + 1
ostatni_wiersz = wiersz - 1


# LIDER I NAJSŁABSZY
najwyzsza = ranking[0][1]
najnizsza = ranking[-1][1]
liderzy = [n for n, s in ranking if s == najwyzsza]
najslabsi = [n for n, s in ranking if s == najnizsza]

ws.cell(ostatni_wiersz + 2, 1).value = "Lider (" + str(najwyzsza) + "): " + ", ".join(liderzy)
ws.cell(ostatni_wiersz + 3, 1).value = "Najsłabszy (" + str(najnizsza) + "): " + ", ".join(najslabsi)
print("Lider:", liderzy, "-", najwyzsza)
print("Najsłabszy:", najslabsi, "-", najnizsza)


# WYKRES
wykres = BarChart()
wykres.type = "bar"                      # słupki poziome
wykres.title = "Ranking menedżerów haseł (wynik W, 0-100)"
wykres.title.overlay = False                   # tytuł nad wykresem, nie na danych
wykres.legend = None

dane = Reference(ws, min_col=2, min_row=START + 1, max_row=ostatni_wiersz)       # kolumna "Suma" z nagłówkiem
nazwy = Reference(ws, min_col=1, min_row=START + 2, max_row=ostatni_wiersz)      # nazwy menedżerów
wykres.add_data(dane, titles_from_data=True)
wykres.set_categories(nazwy)

wykres.x_axis.scaling.orientation = "maxMin"   # lider na górze
wykres.y_axis.crosses = "max"                  # liczby osi na dole (nie przy tytule)
wykres.x_axis.delete = False
wykres.y_axis.delete = False
wykres.dataLabels = DataLabelList()
wykres.dataLabels.showVal = True               # liczby na słupkach
wykres.dataLabels.showSerName = False          # bez nazwy serii
wykres.dataLabels.showCatName = False          # bez nazwy menedżera
wykres.dataLabels.showLegendKey = False        # bez kwadracika legendy
wykres.dataLabels.showPercent = False
wykres.dataLabels.position = "outEnd"          # liczby za końcem słupka
wykres.y_axis.scaling.min = 0
wykres.y_axis.scaling.max = 100
wykres.gapWidth = 60                           # węższe odstępy, grubsze słupki
wykres.plot_area.layout = Layout(manualLayout=ManualLayout(xMode="edge", yMode="edge", x=0.22, y=0.16, w=0.72, h=0.72))   # obszar wykresu niżej, pod tytułem
wykres.width = 24
wykres.height = 14                             # wyższy wykres = nazwy się nie nakładają

ws.add_chart(wykres, "D" + str(START))


# zapis
nowy_plik = "menedzer_hasel_wyniki.xlsx"
wb.save(nowy_plik)
files.download(nowy_plik)
print("Gotowe! Plik został pobrany.")

Saving menedżer haseł.xlsx to menedżer haseł.xlsx
Cena planu płatnego (rocznie) -> min = 19.8 , max = 59.88
Ilość platform obsługujących -> min = 3 , max = 6
Historia wycieków -> min = 1 , max = 5
Niezależne audyty bezpieczeństwa -> min = 2 , max = 5
Łatwość przenoszenia danych do innego menedżera -> min = 3 , max = 5
Udostępnianie haseł innym osobom -> min = 1 , max = 5
Limit urządzeń w darmowej wersji -> min = 1 , max = 5
Passkeys -> min = 1 , max = 5
Suma wag: 1.0
Lider: ['Bitwarden'] - 90.0
Najsłabszy: ['Dashlane'] - 37.2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gotowe! Plik został pobrany.
